In [2]:
from groq import Groq
import requests

In [3]:

client = Groq()

def llm(prompt):
    response = client.chat.completions.create(
    model="openai/gpt-oss-120b",  # or any other Groq model
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ])
    return response.choices[0].message.content

In [4]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url).json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in response:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    # extend adds the iterable data itself inside list, tuple, etc..
    documents.extend(course_data)

len(documents)


1406

In [5]:
[documents[i] for i in range(0,3)]

[{'id': '0e38656cfb',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How do I submit homework?',
  'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"},
 {'id': '226a4baf2f',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What’s new in the 2025 edition?',
  'answer': '- Deployment module updated to **FastAPI** (replacing Flask) and new tools.\n- Neural networks taught with **PyTorch** (theory videos in Keras are kept; an additional PyTorch

In [6]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [7]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [8]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [9]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [10]:
question = "How can i get a certificate?"


In [17]:
USER_PROMPT_TEMPLATE = """
Question: {question}
Context: {context}
"""

In [18]:
search_results = search(question)

prompt = build_prompt(question, search_results)
print(prompt)


Question: How can i get a certificate?
Context: General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced mode, but project submission and
peer review must happen while a live cohort is accepting them.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a t

In [ ]:
answer = llm(prompt)
print(answer)

To receive a certificate, you need to **submit your project before the submission deadline** (i.e., while the course is still accepting submissions). Once your project is submitted and accepted, you’ll be eligible for the certificate.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
documents = []
